# 第 9 章 · 性能分析：roofline、正确计时与 Nsight

本章回答：一个 kernel 是慢在"算"还是慢在"搬"？怎么用一条公式判断它最快能跑多快？怎么正确地测 kernel 时间？Nsight Systems 和 Nsight Compute 各看什么？

配套讲义：仓库里 `cuda-programming/notes/lessons/09-roofline-profiling.md`（讲解、动画、练习都在讲义里）。对应原文：Best Practices §9.1 Timing、§9.2 Bandwidth（`materials/text/cuda-c-best-practices-guide.txt` 第 900–1084 行）；PG §2.5.3.3 Timing Operations in CUDA Streams（`text/cuda-programming-guide.txt` 第 6483–6531 行）；Boehm SGEMM 开头的上限估算（`text/blog-boehm-cuda-matmul-worklog.txt` 第 125–135 行）；GPU MODE L8（`text/gpu-mode-lecture-08-cuda-performance-checklist.txt`）；Nsight Compute guide（`docs/nsight-compute-profiling-guide-v2026.3.1.txt`）；[04 调研](../04-resources-and-optimization-survey.md) §4

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import time

import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 脊点 | 989 / 3.35 | `ridge(**H100_BF16)` | 295 FLOP/byte |
| 第二步 四个 kernel | FLOP ÷ 字节 | `intensity_vec_add_fp32()` 等 + `bound(...)` | 0.083、0.25 → memory；683、1365 → compute |
| 第二步 向量加法上限 | 强度 × 带宽 | `attainable(1/12, **H100_BF16)` | 0.28 TFLOP/s，峰值的 0.028% |
| 第二步 Boehm 的账 | 137 GFLOP ÷ 30 T；268 MB ÷ 768 G | `boehm_lower_bounds()` | 4.57 ms、0.35 ms（Boehm 写 4.5、0.34） |
| 第三步 计时 | 预热 + 中位数 + 有效带宽 | `time_it(fn)`、`effective_bandwidth(8n, 4n, t)` | 因机器而异 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `A6000_FP32`**（来自 `ch09_roofline.py`）

In [ ]:
A6000_FP32 = {"peak": 30e12, "bw": 768e9}

**常量 `H100_BF16`**（来自 `ch09_roofline.py`）

In [ ]:
H100_BF16 = {"peak": 989e12, "bw": 3.35e12}

**函数 `attainable`**（来自 `ch09_roofline.py`）：可达性能 = min(峰值算力, 算术强度 × 带宽)。

In [ ]:
def attainable(intensity: float, peak: float, bw: float) -> float:
    """可达性能 = min(峰值算力, 算术强度 × 带宽)。"""
    return min(peak, intensity * bw)

**函数 `boehm_lower_bounds`**（来自 `ch09_roofline.py`）：Boehm 的估算：算要多久、搬最少的数据要多久（毫秒）。搬 = 读 A、B、C + 写 C，共 4 × N² × 4 B。

In [ ]:
def boehm_lower_bounds(n: int = 4092) -> tuple[float, float]:
    """Boehm 的估算：算要多久、搬最少的数据要多久（毫秒）。搬 = 读 A、B、C + 写 C，共 4 × N² × 4 B。"""
    flops = 2 * n**3 + n**2
    moved = 4 * n**2 * 4
    return flops / A6000_FP32["peak"] * 1e3, moved / A6000_FP32["bw"] * 1e3

**函数 `ridge`**（来自 `ch09_roofline.py`）：脊点 = 峰值算力 ÷ 显存带宽（FLOP / byte）。

In [ ]:
def ridge(peak: float, bw: float) -> float:
    """脊点 = 峰值算力 ÷ 显存带宽（FLOP / byte）。"""
    return peak / bw

**函数 `bound`**（来自 `ch09_roofline.py`）：下面要用到的小工具。

In [ ]:
def bound(intensity: float, peak: float, bw: float) -> str:
    return "memory-bound" if intensity < ridge(peak, bw) else "compute-bound"

**函数 `effective_bandwidth`**（来自 `ch09_roofline.py`）：有效带宽 = (读字节 + 写字节) ÷ 耗时（Best Practices §9.2.2）。

In [ ]:
def effective_bandwidth(read_bytes: int, write_bytes: int, seconds: float) -> float:
    """有效带宽 = (读字节 + 写字节) ÷ 耗时（Best Practices §9.2.2）。"""
    return (read_bytes + write_bytes) / seconds

**函数 `intensity_gemm`**（来自 `ch09_roofline.py`）：N × N 矩阵乘：2N³ FLOP ÷ 至少 3 个矩阵 × N² × 每元素字节。

In [ ]:
def intensity_gemm(n: int, bytes_per: int) -> float:
    """N × N 矩阵乘：2N³ FLOP ÷ 至少 3 个矩阵 × N² × 每元素字节。"""
    return 2 * n**3 / (3 * bytes_per * n**2)

**函数 `intensity_relu_bf16`**（来自 `ch09_roofline.py`）：n 次比较，读 1 个、写 1 个 BF16：n / 4n。

In [ ]:
def intensity_relu_bf16() -> float:
    """n 次比较，读 1 个、写 1 个 BF16：n / 4n。"""
    return 1 / 4

**函数 `intensity_vec_add_fp32`**（来自 `ch09_roofline.py`）：n 次加法，读 2 个、写 1 个 float：n / 12n。

In [ ]:
def intensity_vec_add_fp32() -> float:
    """n 次加法，读 2 个、写 1 个 float：n / 12n。"""
    return 1 / 12

**函数 `time_it`**（来自 `ch09_roofline.py`）：计时规范：先预热，再重复多次取中位数（秒）。GPU 上要换成 CUDA event。

In [ ]:
def time_it(fn, warmup: int = 3, repeat: int = 20) -> float:
    """计时规范：先预热，再重复多次取中位数（秒）。GPU 上要换成 CUDA event。"""
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 H100 BF16 的脊点")
print(f"  脊点 = 989 / 3.35 ≈ {ridge(**H100_BF16):.0f} FLOP/byte")

In [ ]:
print("\n第二步 四个 kernel 放到 roofline 上")
rows = [("向量加法 FP32", intensity_vec_add_fp32()), ("ReLU BF16", intensity_relu_bf16()),
        ("矩阵乘 FP32 N=4096", intensity_gemm(4096, 4)), ("矩阵乘 BF16 N=4096", intensity_gemm(4096, 2))]
for name, ai in rows:
    print(f"  {name:<18} 强度 {ai:8.3f}  → {bound(ai, **H100_BF16)}")
cap = attainable(intensity_vec_add_fp32(), **H100_BF16)
print(f"  向量加法上限 = {cap / 1e12:.2f} TFLOP/s，是峰值的 {cap / H100_BF16['peak']:.3%}")

In [ ]:
print("\n第二步 Boehm 的 A6000（FP32 30 TFLOP/s、768 GB/s）")
print(f"  脊点 = {ridge(**A6000_FP32):.0f}")
t_calc, t_mem = boehm_lower_bounds()
print(f"  4092² 矩阵乘：算 ≈ {t_calc:.2f} ms，搬 268 MB ≈ {t_mem:.2f} ms → compute-bound")

In [ ]:
print("\n第三步 计时规范（CPU 上的 NumPy 加法演示，数字因机器而异）")
n = 4_000_000
a, b = np.ones(n, np.float32), np.ones(n, np.float32)
c = np.empty_like(a)
t = time_it(lambda: np.add(a, b, out=c))
print(f"  中位数耗时 {t * 1e3:.2f} ms，有效带宽 ≈ {effective_bandwidth(8 * n, 4 * n, t) / 1e9:.1f} GB/s")

## 3. 代码里才看得到的

1. **Boehm 的 4.5 ms、0.34 ms 是截断的**：代码算出 4.57 ms 和 0.349 ms，测试按 ±0.1 / ±0.01 的容差核对。268 MB 里包括读 C（因为公式有 β·C）和写 C，一共 4 × 4092² × 4 字节。
2. **CPU 上的 `time.perf_counter` 不需要同步**：NumPy 的加法是同步执行的。GPU 上要换成 CUDA event，或者计时前后都 `cudaDeviceSynchronize()`（第 4 章）。
3. **CPU 算出的"有效带宽"会受缓存影响**：数组小到能放进 CPU 缓存时，测出来比内存带宽高——和讲义说的"小输入要清 L2"是同一个坑。

## 4. 自己改一改

- 把 `n = 4_000_000` 改成 4_000 和 40_000_000，看测出来的"有效带宽"怎么变（小数组全在缓存里）。
- 用 `attainable` 把 A6000 FP32 的 roofline 画几个点：强度 1、10、39、100。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 先分三类：算不过来、搬不过来、GPU 在闲着；
> 算术强度 = FLOP ÷ 字节，脊点 = 算力 ÷ 带宽；低于脊点看带宽，高于脊点看算力；
> 计时用 event，先预热，取中位数，小输入清 L2；
> 先 nsys 看全局，再 ncu 看单个：SOL 两个数，warp 在等什么。